# Preview: upcoming dataset-tools features

> These are **small, concrete** draft PRs with high certainty of landing roughly as shown: the pydantic dataset specs (#1528, already released), non-dask preprocessing backends (#1579), user metadata extraction (#1600), and adaptive/resizable steps (#1601).
>
> Every cell is **guarded** (detected by import/signature introspection, never by version number), so this notebook is safe to "Run All" in the default environment. To run the live demos, use the preview environment:
>
> ```bash
> pixi run -e preview jupyter lab
> ```
>
> The larger, more experimental `coffea.compute` execution refactor lives separately in **`coffea-06-experimental.ipynb`**.

In [ ]:
# Feature detection for the dataset-tools previews. Each is detected by import /
# signature introspection (never by version number), so this notebook is safe to
# "Run All" in the default environment -- missing features just print a note.
import importlib.util
import inspect
from pathlib import Path

import coffea
from coffea.dataset_tools import preprocess as _preprocess


def _has_param(func, name):
    try:
        return name in inspect.signature(func).parameters
    except (TypeError, ValueError):
        return False


HAS_PP_BACKENDS = _has_param(_preprocess, "backend")             # draft PR #1579
HAS_PP_METADATA = _has_param(_preprocess, "metadata_extractor")  # draft PR #1600
HAS_MUTABLE_STEPS = importlib.util.find_spec("coffea.dataset_tools.mutable_steps") is not None  # draft PR #1601


def preview_note(feature, pr):
    print(
        f"[preview] '{feature}' is not in this coffea build ({coffea.__version__}).\n"
        f"          It ships in draft PR {pr}. Launch the preview environment to try it:\n"
        f"            pixi run -e preview jupyter lab"
    )


# A small, network-free sample so the demos run wherever this repo is checked out.
_preview_file = Path("../columnar/data/SMHiggsToZZTo4L.root")
_preview_fileset = {"demo": {"files": {str(_preview_file): "Events"}, "metadata": {"xsec": 1.0}}}

print(f"coffea {coffea.__version__}")
for _flag in ["HAS_PP_BACKENDS", "HAS_PP_METADATA", "HAS_MUTABLE_STEPS"]:
    print(f"  {_flag} = {globals()[_flag]}")

### 1. Pydantic dataset specifications

*Released in coffea 2026.7 (PR #1528) — the foundation the previews build on.*

Filesets can now be expressed as validated `pydantic` models (`DataGroupSpec` / `DatasetSpec` / `ROOTFileSpec`, ...). Malformed filesets fail fast with clear errors, and the models carry form and metadata around for the tools below.

In [ ]:
# 1. Pydantic dataset specifications (released in coffea 2026.7, PR #1528)
# The classic "dict-in / dict-out" fileset still works, but datasets can now be
# expressed as *validated* pydantic models, catching malformed filesets early.
from coffea.dataset_tools import ModelFactory, DatasetSpec

spec = ModelFactory.dict_to_datasetspec(_preview_fileset["demo"])
print("type:", type(spec).__name__, "| is DatasetSpec:", isinstance(spec, DatasetSpec))
print("validated metadata:", dict(spec.metadata))
print("file specs:", [type(fs).__name__ for fs in spec.files.values()])
# round-trip back to a plain dict when a legacy API needs one
_roundtrip = ModelFactory.datasetspec_to_dict(spec)

### 2. Non-dask preprocessing backends — draft PR #1579

Released `preprocess()` builds a **dask-awkward** graph to discover file chunks. PR #1579 adds a `backend=` switch (`"iterative"`, `"futures"`, `"dask"`) so preprocessing can run with **no dask dependency**, plus a dedicated `preprocess_rntuple()` for RNTuple inputs. The backend classes (`IterativeBackend`, `FuturesBackend`, ...) are explicitly designed to plug into the `coffea.compute` refactor below.

In [ ]:
# 2. Non-dask preprocessing backends  (draft PR #1579)
from coffea.dataset_tools import preprocess

if HAS_PP_BACKENDS and _preview_file.exists():
    available, report = preprocess(
        _preview_fileset,
        step_size=50_000,
        save_form=False,
        backend="iterative",  # or "futures"; "dask" reproduces the legacy path
        skip_bad_files=True,
    )
    finfo = list(available["demo"]["files"].values())[0]
    print("preprocessed with the dask-free 'iterative' backend")
    print("  steps discovered:", finfo["steps"])
    print("  num_entries:", finfo["num_entries"])
elif not HAS_PP_BACKENDS:
    preview_note("preprocess(backend=...)", "#1579")
else:
    print("[preview] sample file not found; skipping the live run.")

### 3. User-supplied metadata extraction — draft PR #1600

Computing per-dataset quantities such as the sum of generator weights normally means an extra pass over the files. PR #1600 adds `metadata_extractor` (called once per file on the open handle) and `metadata_reducer` (called once per dataset) hooks to `preprocess()`, folding that work into the preprocessing pass.

In [ ]:
# 3. User-supplied metadata extraction during preprocessing  (draft PR #1600)
from coffea.dataset_tools import preprocess

if HAS_PP_METADATA and _preview_file.exists():
    def per_file(file_handle):
        # runs once per file, on the open uproot file handle
        return {"nentries": int(file_handle["Events"].num_entries)}

    def per_dataset(per_file_meta):
        # reduce the per-file dicts into dataset-level metadata
        return {"nentries_total": sum(m["nentries"] for m in per_file_meta.values())}

    available, _ = preprocess(
        _preview_fileset,
        step_size=50_000,
        save_form=False,
        backend="iterative",
        metadata_extractor=per_file,
        metadata_reducer=per_dataset,
        skip_bad_files=True,
    )
    print("dataset metadata after extraction:", dict(available["demo"]["metadata"]))
elif not HAS_PP_METADATA:
    preview_note("preprocess(metadata_extractor=..., metadata_reducer=...)", "#1600")
else:
    print("[preview] sample file not found; skipping the live run.")

### 4. Adaptive / resizable steps — draft PR #1601

Fixed step sizes over- or under-shoot when chunk cost varies. This **prototype** adds a resizable step generator whose size can be renegotiated mid-stream through the generator `.send()` channel (the same channel `coffea.compute`'s `Computable.gen_steps` uses), plus a `run_adaptive_steps` driver governed by a `WallTimeStepPolicy`. The API is explicitly marked unstable.

In [ ]:
# 4. Adaptive / resizable steps  (draft PR #1601, prototype -- API may change)
if HAS_MUTABLE_STEPS:
    from coffea.dataset_tools.mutable_steps import resizable_steps

    gen = resizable_steps(0, 1_000, 200)
    produced = [next(gen)]
    try:
        while True:
            # after the first chunk, ask the generator to shrink the step to 100
            produced.append(gen.send(100))
    except StopIteration:
        pass
    print("resizable_steps, shrunk mid-stream via .send(100):")
    print(" ", produced)

    # Higher-level driver, operating on a preprocessed pydantic DatasetSpec:
    print(
        "\nHigher-level API (illustrative):\n"
        "    from coffea.dataset_tools.mutable_steps import (\n"
        "        iter_dataset_steps, run_adaptive_steps, WallTimeStepPolicy)\n"
        "    policy = WallTimeStepPolicy(target_seconds=30)\n"
        "    total = run_adaptive_steps(dataset_spec, work_fn, step_size=100_000, policy=policy)"
    )
else:
    preview_note("coffea.dataset_tools.mutable_steps", "#1601")